# Notebook 05b · GRU4Rec
**Input :**
```
F:\mmd\data\features\gru4rec_train.pkl
F:\mmd\data\features\gru4rec_val.pkl
F:\mmd\data\features\gru4rec_test.pkl
F:\mmd\data\features\feature_config.json
```
**Output:**
```
F:\mmd\models\gru4rec\
    best_model.pt           ← checkpoint tốt nhất (theo NDCG@10 trên val)
    last_model.pt
    training_log.json
    gru4rec_results.json    ← Hit@K, NDCG@K trên test set
```
---
### Architecture
```
Input sequence  [PAD, PAD, i1, i2, i3, ..., iT]   shape (B, MAX_SEQ_LEN)
        │
    Embedding    (N_ITEMS, EMBED_DIM)  padding_idx=0
        │
    GRU          (EMBED_DIM → HIDDEN_SIZE, n_layers=1)
        │  take last non-pad hidden state
    Dropout
        │
    Linear       (HIDDEN_SIZE → N_ITEMS)
        │
    logits over all items  →  BPR-max loss
```

## 0 · Imports & config

In [1]:
# !pip install torch torchvision
import gc, json, pickle, time, warnings
from pathlib import Path
import numpy as np
import psutil
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm
warnings.filterwarnings("ignore")

def ram():
    v = psutil.virtual_memory()
    return f"RAM {v.used/1e9:.1f}/{v.total/1e9:.1f} GB ({v.percent:.0f}%)"

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device  : {DEVICE}")
if DEVICE.type == "cuda":
    print(f"GPU     : {torch.cuda.get_device_name(0)}")
    print(f"VRAM    : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")

# # ── Paths ──────────────────────────────────────────────────────────────
# ROOT_DIR    = Path(r"F:\amazon_data")
# FEATURE_DIR = ROOT_DIR / "data" / "features"
# MODEL_DIR   = ROOT_DIR / "models" / "gru4rec"
# MODEL_DIR.mkdir(parents=True, exist_ok=True)
# Tên thư mục 'amazon-features' phải khớp với tên Dataset bạn đặt trên Kaggle
FEATURE_DIR = Path("/kaggle/input/datasets/yagi19611/amazon-features/amazon_feature") 

# Output sẽ được ghi hết vào /kaggle/working để giữ cấu trúc giống hệt máy bạn
WORKING_ROOT = Path("/kaggle/working")
MODEL_DIR    = WORKING_ROOT / "models" / "gru4rec"
FIGURES_DIR  = WORKING_ROOT / "outputs" / "figures"

# Tạo sẵn thư mục lưu output
MODEL_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
# ── Load config ────────────────────────────────────────────────────────
with open(FEATURE_DIR / "feature_config.json") as f:
    CFG = json.load(f)

N_ITEMS     = CFG["N_ITEMS"]
MAX_SEQ_LEN = CFG["MAX_SEQ_LEN"]
PAD_IDX     = CFG["PAD_IDX"]
RANDOM_SEED = CFG["RANDOM_SEED"]
torch.manual_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

# ── Hyperparameters ────────────────────────────────────────────────────
EMBED_DIM    = CFG["embed_dim"]     # 128
HIDDEN_SIZE  = CFG["gru_hidden"]    # 256
N_LAYERS     = CFG["gru_layers"]    # 1
DROPOUT      = 0.3
BATCH_SIZE   = 1024               # giảm xuống 256 nếu OOM
LR           =1e-3
N_EPOCHS     = 20
N_NEG_TRAIN  = 50                   # negative samples lúc train (BPR)
PATIENCE     = 3                    # early stopping patience
TOP_K_LIST   = [5, 10, 20]

print(f"N_ITEMS     : {N_ITEMS:,}")
print(f"EMBED_DIM   : {EMBED_DIM}")
print(f"HIDDEN_SIZE : {HIDDEN_SIZE}")
print(f"BATCH_SIZE  : {BATCH_SIZE}")
print(ram())

Device  : cuda
GPU     : Tesla T4
VRAM    : 15.6 GB
N_ITEMS     : 667,278
EMBED_DIM   : 128
HIDDEN_SIZE : 256
BATCH_SIZE  : 1024
RAM 1.1/33.7 GB (4%)


## 1 · Dataset & DataLoader

In [2]:
import pandas as pd

class SeqDataset(Dataset):
    """Dataset tối ưu RAM nhận trực tiếp Tensor"""
    def __init__(self, inputs_tensor, targets_tensor):
        self.inputs  = inputs_tensor
        self.targets = targets_tensor

    def __len__(self):
        return len(self.targets)

    def __getitem__(self, idx):
        return self.inputs[idx], self.targets[idx]

print("Loading datasets...")
print("  -> Loading train Parquet...")
df_train = pd.read_parquet(FEATURE_DIR / "gru4rec_train.parquet", engine="pyarrow")

# MẸO CỨU RAM: Nếu máy bị OOM (tràn RAM) khi đưa vào PyTorch, hãy mở comment dòng dưới đây 
# để lấy ngẫu nhiên 20% dữ liệu train (khoảng 5-6 triệu mẫu là đủ để mô hình học rất tốt rồi).
# df_train = df_train.sample(frac=0.2, random_state=RANDOM_SEED).reset_index(drop=True)

train_inputs = torch.tensor(df_train["input_seq"].tolist(), dtype=torch.long)
train_targets = torch.tensor(df_train["target"].values, dtype=torch.long)

# Dọn rác DataFrame ngay lập tức
del df_train
gc.collect()

train_ds = SeqDataset(train_inputs, train_targets)

# 2. Load Val & Test set từ Pickles (Vẫn giữ nguyên vì dung lượng nhỏ)
print("  -> Loading val & test Pickles...")
with open(FEATURE_DIR / "gru4rec_val.pkl", "rb") as f:
    val_samples = pickle.load(f)
with open(FEATURE_DIR / "gru4rec_test.pkl", "rb") as f:
    test_samples = pickle.load(f)

val_inputs = torch.tensor([s[0] for s in val_samples], dtype=torch.long)
val_targets = torch.tensor([s[1] for s in val_samples], dtype=torch.long)
val_ds = SeqDataset(val_inputs, val_targets)

test_inputs = torch.tensor([s[0] for s in test_samples], dtype=torch.long)
test_targets = torch.tensor([s[1] for s in test_samples], dtype=torch.long)
test_ds = SeqDataset(test_inputs, test_targets)

del val_samples, test_samples
gc.collect()

# 3. Khởi tạo DataLoader
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                           num_workers=0, pin_memory=(DEVICE.type=="cuda"))
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

print("\n=== DATASET SUMMARY ===")
print(f"Train : {len(train_ds):>10,} samples  |  {len(train_loader):,} batches")
print(f"Val   : {len(val_ds):>10,} samples")
print(f"Test  : {len(test_ds):>10,} samples")
print(ram())

Loading datasets...
  -> Loading train Parquet...
  -> Loading val & test Pickles...

=== DATASET SUMMARY ===
Train : 16,788,005 samples  |  16,395 batches
Val   :  2,712,338 samples
Test  :  2,712,338 samples
RAM 11.1/33.7 GB (34%)


## 2 · Model definition

In [3]:
class GRU4Rec(nn.Module):
    def __init__(self, n_items, embed_dim, hidden_size,
                 n_layers=1, dropout=0.3, pad_idx=0):
        super().__init__()
        self.hidden_size = hidden_size
        self.n_layers    = n_layers
        self.pad_idx     = pad_idx

        self.item_emb = nn.Embedding(n_items, embed_dim, padding_idx=pad_idx)
        self.emb_drop = nn.Dropout(dropout)

        self.gru = nn.GRU(
            input_size  = embed_dim,
            hidden_size = hidden_size,
            num_layers  = n_layers,
            batch_first = True,
            dropout     = dropout if n_layers > 1 else 0.0,
        )
        self.layer_norm = nn.LayerNorm(hidden_size)
        self.out_drop   = nn.Dropout(dropout)
        self.fc         = nn.Linear(hidden_size, n_items, bias=False)

        # Tie weights: output projection = item embedding
        # (chỉ tie nếu embed_dim == hidden_size)
        if embed_dim == hidden_size:
            self.fc.weight = self.item_emb.weight

        self._init_weights()

    def _init_weights(self):
        nn.init.normal_(self.item_emb.weight, std=0.02)
        nn.init.zeros_(self.item_emb.weight[self.pad_idx])
        for name, p in self.gru.named_parameters():
            if "weight" in name:
                nn.init.orthogonal_(p)
            elif "bias" in name:
                nn.init.zeros_(p)

    def forward(self, seq):
        """
        seq : (B, L)  LongTensor, left-padded với pad_idx=0
        return: logits (B, N_ITEMS)
        """
        # Tính seq length thực (số token != PAD) để lấy đúng hidden state
        lengths  = (seq != self.pad_idx).sum(dim=1).clamp(min=1)   # (B,)

        emb      = self.emb_drop(self.item_emb(seq))   # (B, L, E)
        out, _   = self.gru(emb)                       # (B, L, H)

        # Lấy hidden state tại vị trí token thực cuối cùng
        idx      = (lengths - 1).unsqueeze(1).unsqueeze(2).expand(-1, 1, self.hidden_size)
        last_h   = out.gather(1, idx).squeeze(1)       # (B, H)

        last_h   = self.out_drop(self.layer_norm(last_h))
        logits   = self.fc(last_h)                     # (B, N_ITEMS)
        return logits


model = GRU4Rec(
    n_items     = N_ITEMS,
    embed_dim   = EMBED_DIM,
    hidden_size = HIDDEN_SIZE,
    n_layers    = N_LAYERS,
    dropout     = DROPOUT,
    pad_idx     = PAD_IDX,
).to(DEVICE)

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(model)
print(f"\nTrainable params : {n_params:,}")
print(ram())

GRU4Rec(
  (item_emb): Embedding(667278, 128, padding_idx=0)
  (emb_drop): Dropout(p=0.3, inplace=False)
  (gru): GRU(128, 256, batch_first=True)
  (layer_norm): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
  (out_drop): Dropout(p=0.3, inplace=False)
  (fc): Linear(in_features=256, out_features=667278, bias=False)
)

Trainable params : 256,531,712
RAM 11.2/33.7 GB (35%)


## 3 · Loss function — BPR-max

In [4]:
class BPRMaxLoss(nn.Module):
    """
    BPR-max loss (Hidasi & Karatzoglou, 2018):
      L = -log( softmax_neg * sigmoid(s_pos - s_neg) ).mean()
    Ổn định hơn BPR gốc khi có nhiều negatives.
    """
    def __init__(self, n_neg: int = 50, reg: float = 1e-5):
        super().__init__()
        self.n_neg = n_neg
        self.reg   = reg

    def forward(self, logits: torch.Tensor,
                targets: torch.Tensor) -> torch.Tensor:
        """
        logits  : (B, N_ITEMS)
        targets : (B,)  long
        """
        B      = logits.size(0)
        device = logits.device

        # Positive scores
        pos_scores = logits[torch.arange(B, device=device), targets]  # (B,)

        # Sample negatives (uniform — bỏ target index)
        neg_idx = torch.randint(1, N_ITEMS, (B, self.n_neg), device=device)  # (B, n_neg)
        neg_scores = logits.gather(1, neg_idx)                               # (B, n_neg)

        # BPR-max: weight negatives by softmax
        weights    = torch.softmax(neg_scores, dim=1)                        # (B, n_neg)
        diff       = pos_scores.unsqueeze(1) - neg_scores                    # (B, n_neg)
        loss       = -(weights * torch.log(torch.sigmoid(diff) + 1e-8)).sum(dim=1).mean()

        # Regularization on embeddings
        loss = loss + self.reg * (pos_scores ** 2).mean()
        return loss


criterion = BPRMaxLoss(n_neg=N_NEG_TRAIN)
optimizer = optim.Adam(model.parameters(), lr=LR, weight_decay=1e-6)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.5)
print("Loss + optimizer + scheduler defined ✓")

Loss + optimizer + scheduler defined ✓


## 4 · Evaluation function

In [8]:
@torch.no_grad()
def evaluate(model, loader, top_k_list, n_neg=100, desc="Eval"):
    """
    100-way ranking evaluation.
    Với mỗi sample: score target + 100 random negatives,
    tính rank của target trong nhóm này.
    """
    model.eval()
    all_items  = torch.arange(1, N_ITEMS, device=DEVICE)
    max_k      = max(top_k_list)
    hits       = {k: 0 for k in top_k_list}
    ndcgs      = {k: 0.0 for k in top_k_list}
    mrr        = 0.0
    total      = 0

    for seq, target in tqdm(loader, desc=desc, leave=False):
        seq, target = seq.to(DEVICE), target.to(DEVICE)
        logits      = model(seq)           # (B, N_ITEMS)
        B           = logits.size(0)

        # Sample 100 negatives per sample
        neg_idx    = torch.randint(1, N_ITEMS, (B, n_neg), device=DEVICE)
        pos_score  = logits[torch.arange(B, device=DEVICE), target]    # (B,)
        neg_scores = logits.gather(1, neg_idx)                          # (B, 100)

        # Rank of positive among negatives
        ranks = (neg_scores > pos_score.unsqueeze(1)).sum(dim=1) + 1    # (B,) 1-based

        for i in range(B):
            r = int(ranks[i].item())
            for k in top_k_list:
                if r <= k:
                    hits[k]  += 1
                    ndcgs[k] += 1.0 / np.log2(r + 1)
            mrr   += 1.0 / r
            total += 1

    metrics = {f"Hit@{k}":  round(hits[k]/total, 4) for k in top_k_list}
    metrics |= {f"NDCG@{k}": round(ndcgs[k]/total, 4) for k in top_k_list}
    metrics["MRR"]       = round(mrr / total, 4)
    metrics["n_samples"] = total
    return metrics

print("Evaluate function defined ✓")

Evaluate function defined ✓


## 5 · Training loop

In [6]:
# ── Setup & Resume Checkpoint ─────────────────────────────────────────
RESUME_PATH = Path('/kaggle/input/datasets/yagi19611/gru-checkpoint4/checkpoint.pt')
start_epoch = 1
training_log = []
best_ndcg10 = 0.0
patience_cnt = 0

if RESUME_PATH.exists():
    print(f"==> Resuming from checkpoint: {RESUME_PATH}")
    ckpt = torch.load(RESUME_PATH, map_location=DEVICE, weights_only=False)
    
    model.load_state_dict(ckpt["model_state"])
    optimizer.load_state_dict(ckpt["optim_state"])
    scheduler.load_state_dict(ckpt["sched_state"])
    
    start_epoch = ckpt["epoch"] + 1
    best_ndcg10 = ckpt.get("best_ndcg10", 0.0)
    training_log = ckpt.get("training_log", [])
    print(f"==> Tiếp tục train từ Epoch {start_epoch}, Best NDCG@10 cũ: {best_ndcg10:.4f}")

print(f"Training GRU4Rec for {N_EPOCHS} epochs...")
print(f"Device: {DEVICE}  |  Batch: {BATCH_SIZE}  |  LR: {LR}")

# FIX 1: Chạy từ start_epoch thay vì 1
for epoch in range(start_epoch, N_EPOCHS + 1):
    # ── Train ────────────────────────────────────────────────────────
    model.train()
    epoch_loss = 0.0
    t0 = time.time()

    for seq, target in tqdm(train_loader, desc=f"Epoch {epoch:02d} train", leave=False):
        seq, target = seq.to(DEVICE), target.to(DEVICE)
        optimizer.zero_grad()
        logits = model(seq)
        loss   = criterion(logits, target)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
        optimizer.step()
        epoch_loss += loss.item()

    avg_loss = epoch_loss / len(train_loader)
    scheduler.step()

    # ── Validate ─────────────────────────────────────────────────────
    val_metrics = evaluate(model, val_loader, TOP_K_LIST, n_neg=100, desc=f"Epoch {epoch:02d} val")
    elapsed     = time.time() - t0

    log_entry = {"epoch": epoch, "loss": round(avg_loss, 4),
                 "time_s": round(elapsed, 1), **val_metrics}
    training_log.append(log_entry)

    ndcg10 = val_metrics["NDCG@10"]
    print(f"Epoch {epoch:02d} | loss={avg_loss:.4f} | "
          f"Hit@10={val_metrics['Hit@10']:.4f} | "
          f"NDCG@10={ndcg10:.4f} | "
          f"MRR={val_metrics['MRR']:.4f} | "
          f"{elapsed:.0f}s  {ram()}")
          
    # ── Checkpoint & Save best ───────────────────────────────────────
    checkpoint = {
        "epoch": epoch,
        "model_state": model.state_dict(),
        "optim_state": optimizer.state_dict(),
        "sched_state": scheduler.state_dict(),
        "best_ndcg10": max(best_ndcg10, ndcg10), # Giữ best chuẩn xác
        "training_log": training_log,
        "config": CFG
    }

    # Luôn lưu đè vào file checkpoint.pt để resume
    torch.save(checkpoint, MODEL_DIR / "checkpoint.pt")
    
    # FIX 2: Gộp chung logic save best và early stopping thành 1 khối if/else duy nhất
    if ndcg10 > best_ndcg10:
        best_ndcg10 = ndcg10
        patience_cnt = 0
        # Lưu riêng file best
        torch.save(checkpoint, MODEL_DIR / "best_model.pt")
        print(f"  ✓ Saved Best Model (NDCG@10={best_ndcg10:.4f})")
    else:
        patience_cnt += 1
        if patience_cnt >= PATIENCE:
            print(f"  Early stopping at epoch {epoch} (patience={PATIENCE})")
            break

# Save last model + training log
torch.save(model.state_dict(), MODEL_DIR / "last_model.pt")
with open(MODEL_DIR / "training_log.json", "w") as f:
    json.dump(training_log, f, indent=2)
print(f"Training complete. Best NDCG@10 on val: {best_ndcg10:.4f}")

==> Resuming from checkpoint: /kaggle/input/datasets/yagi19611/gru-checkpoint4/checkpoint.pt
==> Tiếp tục train từ Epoch 5, Best NDCG@10 cũ: 0.4001
Training GRU4Rec for 20 epochs...
Device: cuda  |  Batch: 1024  |  LR: 0.001


Epoch 05 train:   0%|          | 0/16395 [00:00<?, ?it/s]

Epoch 05 val:   0%|          | 0/2649 [00:00<?, ?it/s]

Epoch 05 | loss=0.5034 | Hit@10=0.5984 | NDCG@10=0.4003 | MRR=0.3533 | 8421s  RAM 11.9/33.7 GB (37%)
  ✓ Saved Best Model (NDCG@10=0.4003)


Epoch 06 train:   0%|          | 0/16395 [00:00<?, ?it/s]

...

Epoch 08 val:   0%|          | 0/2649 [00:00<?, ?it/s]

Epoch 08 | loss=0.4990 | Hit@10=0.6041 | NDCG@10=0.4040 | MRR=0.3565 | 8399s  RAM 11.9/33.7 GB (37%)


Epoch 09 train:   0%|          | 0/16395 [00:00<?, ?it/s]

Epoch 09 val:   0%|          | 0/2649 [00:00<?, ?it/s]

Epoch 09 | loss=0.4992 | Hit@10=0.6048 | NDCG@10=0.4047 | MRR=0.3571 | 8429s  RAM 11.9/33.7 GB (37%)
  Early stopping at epoch 9 (patience=3)
Training complete. Best NDCG@10 on val: 0.4047


## 6 · Load best model & evaluate on Test set

In [1]:
from pathlib import Path
from torch.utils.data import Dataset, DataLoader
import pickle,  torch
import gc, json, pickle, time, warnings
from pathlib import Path
import numpy as np
import psutil
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm

In [2]:
ROOT_DIR    = Path(r"F:\amazon_data")
MODEL_DIR   = ROOT_DIR / 'models' / 'gru4rec'
FEATURE_DIR  =  ROOT_DIR  / 'data' / 'features'
BATCH_SIZE  = 1024
with open(FEATURE_DIR / "feature_config.json") as f:
    CFG = json.load(f)

N_ITEMS     = CFG["N_ITEMS"]
MAX_SEQ_LEN = CFG["MAX_SEQ_LEN"]
PAD_IDX     = CFG["PAD_IDX"]
RANDOM_SEED = CFG["RANDOM_SEED"]
torch.manual_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

# ── Hyperparameters ────────────────────────────────────────────────────
EMBED_DIM    = CFG["embed_dim"]     # 128
HIDDEN_SIZE  = CFG["gru_hidden"]    # 256
N_LAYERS     = CFG["gru_layers"]    # 1
DROPOUT      = 0.3
BATCH_SIZE   = 1024               # giảm xuống 256 nếu OOM
LR           =1e-3
N_EPOCHS     = 20
N_NEG_TRAIN  = 50                   # negative samples lúc train (BPR)
PATIENCE     = 3                    # early stopping patience
TOP_K_LIST   = [5, 10, 20]

In [3]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [4]:
class GRU4Rec(nn.Module):
    def __init__(self, n_items, embed_dim, hidden_size,
                 n_layers=1, dropout=0.3, pad_idx=0):
        super().__init__()
        self.hidden_size = hidden_size
        self.n_layers    = n_layers
        self.pad_idx     = pad_idx

        self.item_emb = nn.Embedding(n_items, embed_dim, padding_idx=pad_idx)
        self.emb_drop = nn.Dropout(dropout)

        self.gru = nn.GRU(
            input_size  = embed_dim,
            hidden_size = hidden_size,
            num_layers  = n_layers,
            batch_first = True,
            dropout     = dropout if n_layers > 1 else 0.0,
        )
        self.layer_norm = nn.LayerNorm(hidden_size)
        self.out_drop   = nn.Dropout(dropout)
        self.fc         = nn.Linear(hidden_size, n_items, bias=False)

        # Tie weights: output projection = item embedding
        # (chỉ tie nếu embed_dim == hidden_size)
        if embed_dim == hidden_size:
            self.fc.weight = self.item_emb.weight

        self._init_weights()

    def _init_weights(self):
        nn.init.normal_(self.item_emb.weight, std=0.02)
        nn.init.zeros_(self.item_emb.weight[self.pad_idx])
        for name, p in self.gru.named_parameters():
            if "weight" in name:
                nn.init.orthogonal_(p)
            elif "bias" in name:
                nn.init.zeros_(p)

    def forward(self, seq):
        """
        seq : (B, L)  LongTensor, left-padded với pad_idx=0
        return: logits (B, N_ITEMS)
        """
        # Tính seq length thực (số token != PAD) để lấy đúng hidden state
        lengths  = (seq != self.pad_idx).sum(dim=1).clamp(min=1)   # (B,)

        emb      = self.emb_drop(self.item_emb(seq))   # (B, L, E)
        out, _   = self.gru(emb)                       # (B, L, H)

        # Lấy hidden state tại vị trí token thực cuối cùng
        idx      = (lengths - 1).unsqueeze(1).unsqueeze(2).expand(-1, 1, self.hidden_size)
        last_h   = out.gather(1, idx).squeeze(1)       # (B, H)

        last_h   = self.out_drop(self.layer_norm(last_h))
        logits   = self.fc(last_h)                     # (B, N_ITEMS)
        return logits


model = GRU4Rec(
    n_items     = N_ITEMS,
    embed_dim   = EMBED_DIM,
    hidden_size = HIDDEN_SIZE,
    n_layers    = N_LAYERS,
    dropout     = DROPOUT,
    pad_idx     = PAD_IDX,
).to(DEVICE)

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(model)
print(f"\nTrainable params : {n_params:,}")


GRU4Rec(
  (item_emb): Embedding(667278, 128, padding_idx=0)
  (emb_drop): Dropout(p=0.3, inplace=False)
  (gru): GRU(128, 256, batch_first=True)
  (layer_norm): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
  (out_drop): Dropout(p=0.3, inplace=False)
  (fc): Linear(in_features=256, out_features=667278, bias=False)
)

Trainable params : 256,531,712


In [5]:
import pandas as pd

class SeqDataset(Dataset):
    """Dataset tối ưu RAM nhận trực tiếp Tensor"""
    def __init__(self, inputs_tensor, targets_tensor):
        self.inputs  = inputs_tensor
        self.targets = targets_tensor

    def __len__(self):
        return len(self.targets)

    def __getitem__(self, idx):
        return self.inputs[idx], self.targets[idx]

In [6]:

print('start :')
with open(FEATURE_DIR / "gru4rec_test.pkl", "rb") as f:
    test_samples = pickle.load(f)
test_inputs = torch.tensor([s[0] for s in test_samples], dtype=torch.long)
test_targets = torch.tensor([s[1] for s in test_samples], dtype=torch.long)
test_ds = SeqDataset(test_inputs, test_targets)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

print(f"Test  : {len(test_ds):>10,} samples")

del test_samples

start :
Test  :  2,712,338 samples


In [ ]:
# Load best checkpoint
ckpt = torch.load(MODEL_DIR / "best_model.pt", map_location=DEVICE, weights_only=False)
model.load_state_dict(ckpt["model_state"])
print(f"Loaded best checkpoint from epoch {ckpt['epoch']}")
print(f"Val metrics at best epoch: 0.4047")

# Test evaluation
print("\nEvaluating on test set...")
test_metrics = evaluate(model, test_loader, TOP_K_LIST, n_neg=100, desc="Test")

print("\nTest metrics (100-way ranking):")
for k, v in test_metrics.items():
    print(f"  {k:<12}: {v}")

# Lưu kết quả
results = {
    "model": "GRU4Rec",
    "hyperparams": {
        "embed_dim":   EMBED_DIM,
        "hidden_size": HIDDEN_SIZE,
        "n_layers":    N_LAYERS,
        "dropout":     DROPOUT,
        "batch_size":  BATCH_SIZE,
        "lr":          LR,
        "n_neg_train": N_NEG_TRAIN,
        "n_epochs":    ckpt["epoch"],
        "loss":        "BPR-max",
    },
    "best_val_metrics" : ckpt["val_metrics"],
    "test_metrics"     : test_metrics,
}
with open(MODEL_DIR / "gru4rec_results.json", "w") as f:
    json.dump(results, f, indent=2)
print(f"\nResults saved → {MODEL_DIR / 'gru4rec_results.json'}")

Loaded best checkpoint from epoch 6
Val metrics at best epoch: 0.4047

Evaluating on test set...


Test:   0%|          | 0/2649 [00:09<?, ?it/s]

## 7 · Plot training curves

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

FIGURES_DIR = ROOT_DIR / "outputs" / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

log_df = pd.DataFrame(training_log)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(log_df["epoch"], log_df["loss"], marker="o", ms=4, color="#4C72B0")
axes[0].set_title("Training Loss (BPR-max)"); axes[0].set_xlabel("Epoch")

for k, c in zip(TOP_K_LIST, ["#4C72B0","#DD8452","#55A868"]):
    axes[1].plot(log_df["epoch"], log_df[f"Hit@{k}"], marker="o", ms=3,
                 label=f"Hit@{k}", color=c)
axes[1].set_title("Hit@K (Val)"); axes[1].set_xlabel("Epoch"); axes[1].legend()

for k, c in zip(TOP_K_LIST, ["#4C72B0","#DD8452","#55A868"]):
    axes[2].plot(log_df["epoch"], log_df[f"NDCG@{k}"], marker="o", ms=3,
                 label=f"NDCG@{k}", color=c)
axes[2].set_title("NDCG@K (Val)"); axes[2].set_xlabel("Epoch"); axes[2].legend()

# Đánh dấu best epoch
best_ep = ckpt["epoch"]
for ax in axes:
    ax.axvline(best_ep, ls="--", color="red", lw=1.5, label=f"best={best_ep}")
    ax.grid(True, alpha=0.3)

plt.suptitle("GRU4Rec Training Curves", fontweight="bold")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "gru4rec_training.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved → {FIGURES_DIR / 'gru4rec_training.png'}")

---
## Notes

| Quyết định | Lý do |
|---|---|
| BPR-max loss | Ổn định hơn BPR gốc, tốt hơn CE khi cần ranking |
| Left-pad + lấy hidden state tại vị trí thực cuối | Không bị ảnh hưởng bởi PAD tokens |
| `LayerNorm` trước output | Giúp train ổn định, giảm vanishing gradient |
| Early stopping patience=3 | Tránh overfit với dataset lớn |
| 100-way ranking eval | Standard benchmark (không phải full ranking để nhanh hơn) |
| `BATCH_SIZE=512` | Giảm xuống 256 nếu RAM 7.9 GB bị OOM với CPU training |

**Nếu không có GPU:** Training sẽ rất chậm (~30–60 phút/epoch với CPU 8 core).  
Giải pháp: giảm `train_samples` bằng cách random sample 10% train set trước.

**→ Notebook 06:** Experiment tracking & so sánh Item2Vec vs GRU4Rec

In [ ]:
!split -b 500M /kaggle/working/models/gru4rec/checkpoint.pt /kaggle/working/models/gru4rec/ckpt_part_